We saw how, given an LRA formula, we can construct a CAD and perform quantifier
alternation using SQL. We now set up a framework around these concepts, where a
user can supply a "legal" SMT-LIB formula and get a truth value out of it.

## Assumptions

We have several requirements for formulas accepted by our framework:

- The formula should be in prenex normal form. This is always possible, but not
  done automatically.
- (In)equalities should be in the format ($a_0 + a_1\cdot x_1 + \ldots < 0$).
  Again, this is not done automatically.

## Introduction

Formulas can be created using our own datastructures, but we also provide
wrappers that accept the [SMT-LIB](https://smt-lib.org/). Let's look at some
code samples doing both.

In [2]:
import util.sqlcad as sqlcad
import util.formula as f
import util.smtlib as smtlib
import duckdb as db

DBNAME = "dbs/cad_generic.db"

`util.formula` exposes our datastructures. We use it specify the density
property: between any two distinct real numbers exists another real number.

We could write it as follows:

$\forall x, y\ .\ (x < y) \rightarrow \exists z (x < z < y)$

However, we have to rewrite this according to the rules specified above. First,
we rewrite the formula to PNF.

$\forall x,y \exists z\ .\ (x < y) \rightarrow (x < z < y)$

Now we rewrite the linear inequalities to the format $a_0 + a_1\cdot x_1 +
\ldots < 0$. Note that variables don't need to be in order in the inequalities.

$\forall x,y \exists z\ .\ (x - y < 0) \rightarrow (x - z < 0 \land z - y < 0)$

This formula we represent as follows:

In [3]:
density = (
    f.Forall("x",
        f.Forall("y",
            f.Exists("z",
                f.Implies(
                    f.InEq('<', f.Sub([f.Var('x'), f.Var('y')])),
                    f.And([
                        f.InEq('<', f.Sub([f.Var('x'), f.Var('z')])),
                        f.InEq('<', f.Sub([f.Var('z'), f.Var('y')]))
                    ])
                )
            )
        )
    )
)

print(density)

∀x ∀y ∃z (x - y < 0) ⟶ ((x - z < 0) ∧ (z - y < 0))


Which we can now evaluate using our sqlcad framework:

In [4]:
with db.connect(DBNAME) as con:
    display(sqlcad.evaluate_formula(con, density))

┌─────────────┬────────┬────────┬────────┐
│ truth_value │   x    │   y    │   z    │
│   boolean   │ double │ double │ double │
├─────────────┼────────┼────────┼────────┤
│ true        │    0.0 │   -1.0 │   -2.0 │
└─────────────┴────────┴────────┴────────┘

Our framework returns the final truth value and an example satisfying
assignment (if there is one).

We can represent this same formula using SMT-LIB, with the same requirements:

In [5]:
smtlib_formula = """
(assert
  (forall ((x Real) (y Real))
  (exists ((z Real))
  (=> (< (- x y) 0)
      (and (< (- x z) 0)
           (< (- z y) 0))))))
"""

smtlib.parse_from_string(smtlib_formula)

∀x ∀y ∃z (x - y < 0) ⟶ ((x - z < 0) ∧ (z - y < 0))

Which we can again evaluate:

In [6]:
with db.connect(DBNAME) as con:
    display(smtlib.evaluate_smtlib_formula_str(con, smtlib_formula))

Evaluating ∀x ∀y ∃z (x - y < 0) ⟶ ((x - z < 0) ∧ (z - y < 0))


┌─────────────┬────────┬────────┬────────┐
│ truth_value │   x    │   y    │   z    │
│   boolean   │ double │ double │ double │
├─────────────┼────────┼────────┼────────┤
│ true        │    0.0 │   -1.0 │   -2.0 │
└─────────────┴────────┴────────┴────────┘

We also allow passing SMT-LIB files instead. For example, a file containing a
random formula with symbols we support:

In [7]:
smtlib_file = "smtlib/all_node_types.smt2"
with open(smtlib_file, 'r', encoding='utf-8') as f:
    print("SMTLIB formule:")
    print(f.read())

print("After parsing:")
smtlib.parse_from_file(smtlib_file)

SMTLIB formule:
(declare-fun F (Real Real) Real)

(assert
    (forall ((x Real))
    (exists ((y Real) (z Real))
        (and (> x 0)
            (= (F x y) 0)
            (< (+ x y -1) 0)
            (>= (+ (* 3.2 x) y) 0)
            (or (not (= z 0)) (= (- y z) 0))))))

After parsing:


∀x ∃y ∃z (x > 0) ∧ (F(x,y) = 0) ∧ (x + y + -1.0 < 0) ∧ (3.2 * x + y >= 0) ∧ ((¬(z = 0)) ∨ (y - z = 0))

## More Examples

Now that we're familiar with the framework, let's start evaluating some more
formulas. We begin extremely simple:

$\forall x\ .\ x + 1 > 0$

In [8]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert (forall ((x Real)) (> (+ x 1) 0)))
    """)
    display(res)

Evaluating ∀x x + 1.0 > 0


┌─────────────┬────────┐
│ truth_value │   x    │
│   boolean   │ double │
├─────────────┼────────┤
│ false       │   NULL │
└─────────────┴────────┘

This yields false, as expected. Let's replace the $\forall$ with an $\exists$,
this should surely be true.

In [9]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert (exists ((x Real)) (> (+ x 1) 0)))
    """)
    display(res)

Evaluating ∃x x + 1.0 > 0


┌─────────────┬────────┐
│ truth_value │   x    │
│   boolean   │ double │
├─────────────┼────────┤
│ true        │    0.0 │
└─────────────┴────────┘

Another basic example with quantifier alternation:

In [10]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert
            (forall ((x Real))
            (exists ((y Real)) (> (+ x y) 0))))
    """)
    display(res)

Evaluating ∀x ∃y x + y > 0


┌─────────────┬────────┬────────┐
│ truth_value │   x    │   y    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ true        │    0.0 │    1.0 │
└─────────────┴────────┴────────┘

The sum of two positive numbers is positive:

In [11]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert
            (forall ((x Real) (y Real))
                (and (> x 0) (> y 0) (> (+ x y) 0))))
    """)
    display(res)

Evaluating ∀x ∀y (x > 0) ∧ (y > 0) ∧ (x + y > 0)


┌─────────────┬────────┬────────┐
│ truth_value │   x    │   y    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ false       │   NULL │   NULL │
└─────────────┴────────┴────────┘

We made a mistake in our formula there; the conjuction should be an implication.
So the false above is indeed correct. With the implication, we expect the
formula to be true.

In [12]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert
            (forall ((x Real) (y Real))
                (=> (and (> x 0) (> y 0))
                    (> (+ x y) 0))))
    """)
    display(res)

Evaluating ∀x ∀y ((x > 0) ∧ (y > 0)) ⟶ (x + y > 0)


┌─────────────┬────────┬────────┐
│ truth_value │   x    │   y    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ true        │   -1.0 │   -1.0 │
└─────────────┴────────┴────────┘

And yet another example: if $x+y$ and $x-y$ are both negative, then surely $x$
is negative.

$\forall x y . x+y < 0 \land x-y<0 \rightarrow x<0$

In [13]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert
          (forall ((x Real) (y Real))
            (=> (and (< (+ y x) 0) (< (- y x) 0))
                (< y 0))))
    """)
    display(res)

Evaluating ∀x ∀y ((y + x < 0) ∧ (y - x < 0)) ⟶ (y < 0)


┌─────────────┬────────┬────────┐
│ truth_value │   x    │   y    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ true        │   -1.0 │   -2.0 │
└─────────────┴────────┴────────┘

And another example, one from the paper:

$\forall x \exists y\ .\ ((x \geq 1) \land (y \geq 1)) \rightarrow (x + y - 3 > 0)$

In [14]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (assert
          (forall ((x Real))
          (exists ((y Real))
            (=> (and (>= (- x 1) 0) (>= (- y 1) 0))
                (> (+ -3 x y) 0)))))
    """)
    display(res)

Evaluating ∀x ∃y ((x - 1.0 >= 0) ∧ (y - 1.0 >= 0)) ⟶ (-3.0 + x + y > 0)


┌─────────────┬────────┬────────┐
│ truth_value │   x    │   y    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ true        │    0.0 │    0.0 │
└─────────────┴────────┴────────┘

## Free Variables

We allow users to have free variables in their input formulas. We simply return
the first assignment that satisfies the formula (if there is any). Because we
work with a cylindrical decomposition, we know which cells are true after
eliminating quantifiers and can simply use their sample points. Behind the
scenes, we prepend existential quantifiers for the free variables to achieve the
same without needing to handle a specific edge case.

We will show a few examples. Note that free variables have to be declared with
`declare-fun` to get valid SMT-LIB formulas.

In [15]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (declare-fun y () Real)

        (assert
          (forall ((x Real))
            (> (- x y) 0)))
    """)
    display(res)

Evaluating ∀x x - y > 0


┌─────────────┬────────┬────────┐
│ truth_value │   y    │   x    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ false       │   NULL │   NULL │
└─────────────┴────────┴────────┘

The formula above is ofcourse not true, let's try $\exists x$ instead.

In [17]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (declare-fun y () Real)

        (assert
          (exists ((x Real))
            (> (- x y) 0)))
    """)
    display(res)

Evaluating ∃x x - y > 0


┌─────────────┬────────┬────────┐
│ truth_value │   y    │   x    │
│   boolean   │ double │ double │
├─────────────┼────────┼────────┤
│ true        │    0.0 │    1.0 │
└─────────────┴────────┴────────┘

We can see that $x=1$ and $y=0$ is a satisfying assignment.

We can have as many free vars as we want.

In [20]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (declare-fun y () Real)
        (declare-fun z () Real)
        (declare-fun u () Real)
        (declare-fun v () Real)
        (declare-fun w () Real)

        (assert
          (exists ((x Real))
            (> (+ x y z u v w) 0)))
    """)
    display(res)

Evaluating ∃x x + y + z + u + v + w > 0


┌─────────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ truth_value │   y    │   z    │   u    │   v    │   w    │   x    │
│   boolean   │ double │ double │ double │ double │ double │ double │
├─────────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ true        │    0.0 │    0.0 │    0.0 │    0.0 │    0.0 │    1.0 │
└─────────────┴────────┴────────┴────────┴────────┴────────┴────────┘

Another small example: $\forall y\ .\ x + y = y$, but using another auxiliary
variable.

In [29]:
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula_str(con, """
        (declare-fun x () Real)

        (assert
          (forall ((y Real))
          (exists ((z Real))
            (and (= (+ x y (- z)) 0)
                 (= (- z y) 0)))))
    """)
    display(res)

Evaluating ∀y ∃z (x + y + -1.0 * z = 0) ∧ (z - y = 0)


┌─────────────┬────────┬────────┬────────┐
│ truth_value │   x    │   y    │   z    │
│   boolean   │ double │ double │ double │
├─────────────┼────────┼────────┼────────┤
│ true        │    0.0 │    0.0 │    0.0 │
└─────────────┴────────┴────────┴────────┘

## Deciding a PWL

Now a segue for our next notebook. Imagine we have a piece-wise linear function
(PWL) as follows:

$
f(x) = \begin{cases} 
    -x & \text{if } x < 0 \\
    x & \text{if } x \geq 0 
\end{cases}
$

And another PWL that is exactly opposite:

$
f'(x) = \begin{cases} 
    x & \text{if } x < 0 \\
    -x & \text{if } x \geq 0 
\end{cases}
$

Is the sum of these functions always 0? To answer this question, we first
translate both functions to LRA:

- $f(x)$: $(x < 0 \rightarrow y + x = 0) \land (x >= 0 \rightarrow y - x = 0)$
- $f'(x)$: $(x < 0 \rightarrow y - x = 0) \land (x >= 0 \rightarrow y + x = 0)$

The full LRA formula then becomes:

$
\forall x \exists y \forall x' \exists y' . \\
    (\\
    \quad (x < 0 \rightarrow y + x = 0) \land (x \geq 0 \rightarrow y - x = 0) \\
    \quad \land (x' < 0 \rightarrow y' - x' = 0) \land (x' \geq 0 \rightarrow y' - x' = 0) \\
    )\\
    \rightarrow (y + y' = 0)
$

In [16]:
# We saved the formula to an SMT-LIB file.
with db.connect(DBNAME) as con:
    res = smtlib.evaluate_smtlib_formula(con, "smtlib/sum_of_pwls.smt2")
    display(res)

Evaluating ∀x1 ∃y1 ∀x2 ∃y2 ((((¬(x1 < 0)) ∨ (y1 + x1 = 0)) ∧ ((¬(x1 >= 0)) ∨ (y1 - x1 = 0))) ∧ (((¬(x2 < 0)) ∨ (y2 - x2 = 0)) ∧ ((¬(x2 >= 0)) ∨ (y2 + x2 = 0)))) ⟶ (¬(y1 + y2 = 0))


┌─────────────┬────────┬────────┬────────┬────────┐
│ truth_value │   x1   │   y1   │   x2   │   y2   │
│   boolean   │ double │ double │ double │ double │
├─────────────┼────────┼────────┼────────┼────────┤
│ true        │   -1.0 │   -2.0 │   -3.0 │   -4.0 │
└─────────────┴────────┴────────┴────────┴────────┘

Now imagine if our framework instead allowed function symbols for LRA. The input
formula then simply becomes $\forall x\ .\ f(x) = f'(x)$. In the next notebook,
we will allow such syntax for functions represented by neural networks, which
will get translated to a lengthy implication such as the one above behind the
scenes.